In [ ]:
PROBLEM LINK : https://www.sparkplayground.com/pyspark-coding-interview-questions/running_payroll

In [ ]:
# SOLUTION 1 => BRUTE FORCE


from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W

spark = SparkSession.builder.appName('Spark Playground').getOrCreate()

# Assume the dataframes employees, payroll are already initialized.


# work <=40hrs => hrs * hrly rate
# work >40hrs => 40*hrly + overtime * 1.5* hrly 
# overtime = F.col("hours_worked") - 40


payroll = payroll.withColumn("overtime" ,F.expr("case when hours_worked<=40 then 0 else hours_worked-40 END"))
payroll = payroll.withColumn("pay" , F.expr("overtime*1.5*hourly_rate + (hours_worked-overtime)*hourly_rate"))
# Write the logic and display the final dataframe
df_result = payroll.join(employees , on= employees.employee_id == payroll.employee_id ).select(payroll.employee_id, "name", "pay", "position")
display(df_result)


In [ ]:
# SOLUTION 2 => USED NATIVE FUNCTION SO OPTIMIZED

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W

spark = SparkSession.builder.appName('Spark Playground').getOrCreate()

# Assume 'employees' and 'payroll' dataframes are already initialized.

# 1. Calculate Overtime using F.when().otherwise() instead of F.expr()
payroll_with_ot = payroll.withColumn(
    "overtime",
    F.when(F.col("hours_worked") <= 40, 0)
     .otherwise(F.col("hours_worked") - 40)
)

# 2. Calculate Pay using pure column arithmetic
payroll_with_pay = payroll_with_ot.withColumn(
    "pay",
    (F.col("overtime") * 1.5 * F.col("hourly_rate")) + 
    ((F.col("hours_worked") - F.col("overtime")) * F.col("hourly_rate"))
)

# 3. Optimized Join & Select
# Pass the column name as a string "employee_id" to prevent duplicate ID columns!
df_result = payroll_with_pay.join(employees, on="employee_id", how="inner") \
                            .select("employee_id", "name", "pay", "position")

display(df_result)